# 2. PYNQ 上板 MNIST 真实推理测试

本 notebook 在 PYNQ-Z2 上运行。

**前提：**
1. 已在宿主机运行 `01_generate_mnist_data.ipynb` 生成 `mnist_real_data.zip`
2. 已上传 `mnist_real_data.zip`、`cim_soc.bit`、`cim_soc.hwh` 到 Jupyter 同目录

**本 notebook 做的事：**
1. 加载 bitstream
2. 一次性加载权重和 bias（所有图片共享）
3. 逐张加载测试图片 → 推理 → 对比 Python golden 结果
4. 统计硬件推理准确率

In [1]:
!unzip -o mnist_real_data.zip

Archive:  mnist_real_data.zip
   creating: mnist_real_data/
   creating: mnist_real_data/test_images/
  inflating: mnist_real_data/quant_params.hex  
  inflating: mnist_real_data/zero_points.hex  
  inflating: mnist_real_data/fc2_bias.hex  
  inflating: mnist_real_data/fc1_bias.hex  
  inflating: mnist_real_data/fc2_weight_tiles.hex  
  inflating: mnist_real_data/fc1_weight_tiles.hex  
  inflating: mnist_real_data/test_images/img_0136_fc2.hex  
  inflating: mnist_real_data/test_images/img_0159_fc2.hex  
  inflating: mnist_real_data/test_images/img_0004_label.txt  
  inflating: mnist_real_data/test_images/img_0176_pred.txt  
  inflating: mnist_real_data/test_images/img_0186_fc2.hex  
  inflating: mnist_real_data/test_images/img_0102.hex  
  inflating: mnist_real_data/test_images/img_0170_fc2.hex  
  inflating: mnist_real_data/test_images/img_0037_pred.txt  
  inflating: mnist_real_data/test_images/img_0173_fc2.hex  
  inflating: mnist_real_data/test_images/img_0027_fc1.hex  
  inflating

  inflating: mnist_real_data/test_images/img_0197_label.txt  
  inflating: mnist_real_data/test_images/img_0079_pred.txt  
  inflating: mnist_real_data/test_images/img_0001_fc1.hex  
  inflating: mnist_real_data/test_images/img_0063_label.txt  
  inflating: mnist_real_data/test_images/img_0076_fc1.hex  
  inflating: mnist_real_data/test_images/img_0049.hex  
  inflating: mnist_real_data/test_images/img_0156_fc2.hex  
  inflating: mnist_real_data/test_images/img_0054_fc2.hex  
  inflating: mnist_real_data/test_images/img_0120.hex  
  inflating: mnist_real_data/test_images/img_0143.hex  
  inflating: mnist_real_data/test_images/img_0065_label.txt  
  inflating: mnist_real_data/test_images/img_0022.hex  
  inflating: mnist_real_data/test_images/img_0122_label.txt  
  inflating: mnist_real_data/test_images/img_0003_fc1.hex  
  inflating: mnist_real_data/test_images/img_0005_label.txt  
  inflating: mnist_real_data/test_images/img_0061_label.txt  
  inflating: mnist_real_data/test_images/im

  inflating: mnist_real_data/test_images/img_0172_fc1.hex  
  inflating: mnist_real_data/test_images/img_0008_label.txt  
  inflating: mnist_real_data/test_images/img_0063_fc2.hex  
  inflating: mnist_real_data/test_images/img_0045_label.txt  
  inflating: mnist_real_data/test_images/img_0126.hex  
  inflating: mnist_real_data/test_images/img_0160_pred.txt  
  inflating: mnist_real_data/test_images/img_0199.hex  
  inflating: mnist_real_data/test_images/img_0142.hex  
  inflating: mnist_real_data/test_images/img_0191_label.txt  
  inflating: mnist_real_data/test_images/img_0145_fc1.hex  
  inflating: mnist_real_data/test_images/img_0170.hex  
  inflating: mnist_real_data/test_images/img_0081_fc1.hex  
  inflating: mnist_real_data/test_images/img_0176_fc1.hex  
  inflating: mnist_real_data/test_images/img_0087_pred.txt  
  inflating: mnist_real_data/test_images/img_0171_fc1.hex  
  inflating: mnist_real_data/test_images/img_0062_fc2.hex  
  inflating: mnist_real_data/test_images/img_012

  inflating: mnist_real_data/test_images/img_0161_label.txt  
  inflating: mnist_real_data/test_images/img_0191_fc1.hex  
  inflating: mnist_real_data/test_images/img_0136_fc1.hex  
  inflating: mnist_real_data/test_images/img_0054.hex  
  inflating: mnist_real_data/test_images/img_0194.hex  
  inflating: mnist_real_data/test_images/img_0047_fc1.hex  
  inflating: mnist_real_data/test_images/img_0185_label.txt  
  inflating: mnist_real_data/test_images/img_0111_pred.txt  
  inflating: mnist_real_data/test_images/img_0170_fc1.hex  
  inflating: mnist_real_data/test_images/img_0117_fc2.hex  
  inflating: mnist_real_data/test_images/img_0068_label.txt  
  inflating: mnist_real_data/test_images/img_0077_fc1.hex  
  inflating: mnist_real_data/test_images/img_0103_pred.txt  
  inflating: mnist_real_data/test_images/img_0056_pred.txt  
  inflating: mnist_real_data/test_images/img_0137_pred.txt  
  inflating: mnist_real_data/test_images/img_0151_fc1.hex  
  inflating: mnist_real_data/test_imag

  inflating: mnist_real_data/test_images/img_0113.hex  
  inflating: mnist_real_data/test_images/img_0108_fc2.hex  
  inflating: mnist_real_data/test_images/img_0062_label.txt  
  inflating: mnist_real_data/test_images/img_0008.hex  
  inflating: mnist_real_data/test_images/img_0042_fc1.hex  
  inflating: mnist_real_data/test_images/img_0072_fc1.hex  
  inflating: mnist_real_data/test_images/img_0140_fc1.hex  
  inflating: mnist_real_data/test_images/img_0070.hex  
  inflating: mnist_real_data/test_images/img_0038.hex  
  inflating: mnist_real_data/test_images/img_0103_label.txt  
  inflating: mnist_real_data/test_images/img_0122.hex  
  inflating: mnist_real_data/test_images/img_0069_label.txt  
  inflating: mnist_real_data/test_images/img_0055_fc2.hex  
  inflating: mnist_real_data/test_images/img_0068.hex  
  inflating: mnist_real_data/test_images/img_0110_fc2.hex  
  inflating: mnist_real_data/test_images/img_0050_fc2.hex  
  inflating: mnist_real_data/test_images/img_0087_label.tx

  inflating: mnist_real_data/test_images/img_0065_fc1.hex  
  inflating: mnist_real_data/test_images/img_0137_fc1.hex  
  inflating: mnist_real_data/test_images/img_0124_fc1.hex  
  inflating: mnist_real_data/test_images/img_0162.hex  
  inflating: mnist_real_data/test_images/img_0082_label.txt  
  inflating: mnist_real_data/test_images/img_0099.hex  
  inflating: mnist_real_data/test_images/img_0014_fc1.hex  
  inflating: mnist_real_data/test_images/img_0165_label.txt  
  inflating: mnist_real_data/test_images/img_0131_fc2.hex  
  inflating: mnist_real_data/test_images/img_0041_fc1.hex  
  inflating: mnist_real_data/test_images/img_0011_pred.txt  
  inflating: mnist_real_data/test_images/img_0009.hex  
  inflating: mnist_real_data/test_images/img_0170_pred.txt  
  inflating: mnist_real_data/test_images/img_0012_fc1.hex  
  inflating: mnist_real_data/test_images/img_0188_pred.txt  
  inflating: mnist_real_data/test_images/img_0152_fc1.hex  
  inflating: mnist_real_data/test_images/img_

In [2]:
from pynq import Overlay, MMIO
import numpy as np
import os, glob

In [3]:
# ====================================================================
# 1. 加载 Bitstream
# ====================================================================
ol = Overlay('cim_soc.bit')
print('Overlay loaded!')

BASE = 0x40000000
mmio = MMIO(BASE, 0x4000)

Overlay loaded!


In [4]:
# ====================================================================
# 2. CSR 地址定义
# ====================================================================
CTRL          = 0x000
STATUS        = 0x004
CSR_IN_DIM    = 0x010
CSR_OUT_DIM   = 0x014
CSR_N_IB      = 0x018
CSR_N_OB      = 0x01C
REQUANT_MULT  = 0x020
REQUANT_SHIFT = 0x024
INPUT_ZP      = 0x028
ACT_MODE      = 0x02C
CYCLE_CNT_LO  = 0x030
MAC_CNT_LO    = 0x038
PRED_CLASS    = 0x040
WDMA_ADDR     = 0x044
WDMA_DATA     = 0x048
WDMA_CTRL     = 0x04C
LOGIT_BASE    = 0x100
MEM_INPUT     = 0x1000
MEM_BIAS      = 0x2000

In [10]:
# ====================================================================
# 3. 硬件操作函数
# ====================================================================
def soft_reset():
    mmio.write(CTRL, 0x4)

def clear_done():
    mmio.write(CTRL, 0x2)

def configure_layer(in_dim, out_dim, zp, mult, shift, act):
    n_ib = (in_dim + 15) // 16
    n_ob = (out_dim + 15) // 16
    mmio.write(CSR_IN_DIM, in_dim)
    mmio.write(CSR_OUT_DIM, out_dim)
    mmio.write(CSR_N_IB, n_ib)
    mmio.write(CSR_N_OB, n_ob)
    mmio.write(REQUANT_MULT, mult)
    mmio.write(REQUANT_SHIFT, shift)
    mmio.write(INPUT_ZP, int(zp) & 0xFFFFFFFF)
    mmio.write(ACT_MODE, act)

def load_weights_burst(chunks):
    mmio.write(WDMA_ADDR, 0)
    mmio.write(WDMA_CTRL, 0x02)
    for c in chunks:
        mmio.write(WDMA_DATA, int(c))
    mmio.write(WDMA_CTRL, 0x00)

def load_bias(bias_list):
    for i, b in enumerate(bias_list):
        mmio.write(MEM_BIAS + 4*i, int(b) & 0xFFFFFFFF)

def load_input(data_u8):
    padded = list(data_u8)
    while len(padded) % 16 != 0:
        padded.append(0)
    for i, x in enumerate(padded):
        mmio.write(MEM_INPUT + 4*i, int(x) & 0xFF)

def run_inference():
    clear_done()
    mmio.write(CTRL, 0x1)
    while not (mmio.read(STATUS) & 0x2):
        pass
    return mmio.read(CYCLE_CNT_LO), mmio.read(MAC_CNT_LO)

def read_output(out_dim):
    out = []
    for i in range(out_dim):
        v = mmio.read(LOGIT_BASE + 4*i)
        out.append(np.uint8(v & 0xFF).view(np.int8))
    return out

print('HW functions loaded.')

HW functions loaded.


In [11]:
# ====================================================================
# 4. Hex 文件读取工具
# ====================================================================
DATA_DIR = 'mnist_real_data'

def read_hex_u32(filepath):
    vals = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line: vals.append(int(line, 16))
    return vals

def read_hex_u8(filepath):
    vals = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line: vals.append(int(line, 16) & 0xFF)
    return vals

def read_hex_int8(filepath):
    return [np.uint8(v).view(np.int8) for v in read_hex_u8(filepath)]

In [12]:
# ====================================================================
# 5. 加载模型参数（权重 + bias + 量化参数）
# ====================================================================
fc1_weight_chunks = read_hex_u32(f'{DATA_DIR}/fc1_weight_tiles.hex')
fc2_weight_chunks = read_hex_u32(f'{DATA_DIR}/fc2_weight_tiles.hex')
fc1_bias = read_hex_u32(f'{DATA_DIR}/fc1_bias.hex')
fc2_bias = read_hex_u32(f'{DATA_DIR}/fc2_bias.hex')
quant_params = read_hex_u32(f'{DATA_DIR}/quant_params.hex')
zero_points = read_hex_u32(f'{DATA_DIR}/zero_points.hex')

fc1_mult, fc1_shift = quant_params[0], quant_params[1]
fc2_mult, fc2_shift = quant_params[2], quant_params[3]
# zero_points are stored as unsigned 32-bit; convert to signed
hw_zp1 = np.int32(np.uint32(zero_points[0]))
hw_zp2 = np.int32(np.uint32(zero_points[1]))

print(f'FC1 weight chunks: {len(fc1_weight_chunks)}')
print(f'FC2 weight chunks: {len(fc2_weight_chunks)}')
print(f'Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})')
print(f'HW zero points: FC1={hw_zp1}, FC2={hw_zp2}')

FC1 weight chunks: 25088
FC2 weight chunks: 512
Quant: fc1=(14,16), fc2=(140,16)
HW zero points: FC1=0, FC2=0


In [13]:
# ====================================================================
# 6. AXI 连通性测试
# ====================================================================
print('=== AXI Connectivity Test ===')
soft_reset()
mmio.write(CSR_IN_DIM, 784)
rb = mmio.read(CSR_IN_DIM)
print(f'  IN_DIM write=784, readback={rb}  {"PASS" if rb==784 else "FAIL"}')
assert rb == 784, 'AXI failed!'

=== AXI Connectivity Test ===
  IN_DIM write=784, readback=784  PASS


In [14]:
# ====================================================================
# 7. 逐张推理测试
# ====================================================================
img_dir = f'{DATA_DIR}/test_images'
img_files = sorted(glob.glob(f'{img_dir}/img_?????.hex') + glob.glob(f'{img_dir}/img_????.hex'))
n_images = len(img_files)
print(f'Found {n_images} test images')

hw_correct = 0
py_match = 0
total = 0
errors = []

for img_path in img_files:
    basename = os.path.basename(img_path).replace('.hex', '')
    
    # Read test data
    img_u8 = read_hex_u8(img_path)
    label = int(open(f'{img_dir}/{basename}_label.txt').read().strip())
    py_pred = int(open(f'{img_dir}/{basename}_pred.txt').read().strip())
    fc1_golden = read_hex_int8(f'{img_dir}/{basename}_fc1.hex')
    fc2_golden = read_hex_int8(f'{img_dir}/{basename}_fc2.hex')
    
    # === FC1: 784→128, ReLU ===
    soft_reset()
    configure_layer(784, 128, hw_zp1, fc1_mult, fc1_shift, 1)
    load_weights_burst(fc1_weight_chunks)
    load_bias(fc1_bias)
    load_input(img_u8)
    cycles1, macs1 = run_inference()
    fc1_out = read_output(128)
    
    # === FC2: 128→10, no activation ===
    configure_layer(128, 10, hw_zp2, fc2_mult, fc2_shift, 0)
    load_weights_burst(fc2_weight_chunks)
    load_bias(fc2_bias)
    fc1_out_u8 = [int(x) & 0xFF for x in fc1_out]
    load_input(fc1_out_u8)
    cycles2, macs2 = run_inference()
    fc2_out = read_output(10)
    
    hw_pred = mmio.read(PRED_CLASS)
    
    # Compare
    fc1_match = all(fc1_out[i] == fc1_golden[i] for i in range(128))
    fc2_match = all(fc2_out[i] == fc2_golden[i] for i in range(10))
    pred_match = (hw_pred == py_pred)
    label_correct = (hw_pred == label)
    
    if label_correct: hw_correct += 1
    if pred_match: py_match += 1
    total += 1
    
    status = '✓' if (fc1_match and fc2_match and pred_match) else '✗'
    label_status = '✓' if label_correct else '✗'
    
    print(f'  {basename}: label={label}, hw_pred={hw_pred}, py_pred={py_pred} '
          f'fc1={"OK" if fc1_match else "MISMATCH"} '
          f'fc2={"OK" if fc2_match else "MISMATCH"} '
          f'bit-exact={status} correct={label_status}')
    
    if not (fc1_match and fc2_match):
        errors.append(basename)
        if not fc1_match:
            diffs = [i for i in range(128) if fc1_out[i] != fc1_golden[i]]
            print(f'    FC1 diffs at {diffs[:5]}...')
        if not fc2_match:
            for i in range(10):
                if fc2_out[i] != fc2_golden[i]:
                    print(f'    FC2[{i}]: hw={fc2_out[i]}, golden={fc2_golden[i]}')

Found 200 test images
  img_0000: label=7, hw_pred=7, py_pred=7 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0001: label=2, hw_pred=2, py_pred=2 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0002: label=1, hw_pred=1, py_pred=1 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0003: label=0, hw_pred=0, py_pred=0 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0004: label=4, hw_pred=4, py_pred=4 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0005: label=1, hw_pred=1, py_pred=1 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0006: label=4, hw_pred=4, py_pred=4 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0007: label=9, hw_pred=9, py_pred=9 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0008: label=5, hw_pred=5, py_pred=5 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0009: label=9, hw_pred=9, py_pred=9 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0010: label=0, hw_pred=0, py_pred=0 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0011: label=6, hw_pred=6, py_pred=6 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0012: label=9, hw_pred=9, py_pred=9 

  img_0105: label=9, hw_pred=9, py_pred=9 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0106: label=2, hw_pred=2, py_pred=2 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0107: label=1, hw_pred=1, py_pred=1 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0108: label=9, hw_pred=9, py_pred=9 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0109: label=4, hw_pred=4, py_pred=4 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0110: label=8, hw_pred=8, py_pred=8 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0111: label=7, hw_pred=7, py_pred=7 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0112: label=3, hw_pred=3, py_pred=3 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0113: label=9, hw_pred=9, py_pred=9 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0114: label=7, hw_pred=7, py_pred=7 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0115: label=4, hw_pred=9, py_pred=9 fc1=OK fc2=OK bit-exact=✓ correct=✗
  img_0116: label=4, hw_pred=4, py_pred=4 fc1=OK fc2=OK bit-exact=✓ correct=✓
  img_0117: label=4, hw_pred=4, py_pred=4 fc1=OK fc2=OK bit-exac

In [15]:
# ====================================================================
# 8. 总结
# ====================================================================
print(f'\n{"="*60}')
print(f'MNIST Real Inference Results')
print(f'{"="*60}')
print(f'  Total images:          {total}')
print(f'  HW accuracy:           {100.*hw_correct/total:.1f}% ({hw_correct}/{total})')
print(f'  Bit-exact match (py):  {100.*py_match/total:.1f}% ({py_match}/{total})')
print(f'  Computation errors:    {len(errors)}')
if len(errors) == 0:
    print(f'\n>>> ALL IMAGES BIT-EXACT MATCH — 硬件计算完全正确 <<<')
    print(f'>>> 真实 MNIST 手写数字识别成功！ <<<')
else:
    print(f'\n  Mismatched images: {errors}')
print(f'{"="*60}')


MNIST Real Inference Results
  Total images:          200
  HW accuracy:           99.5% (199/200)
  Bit-exact match (py):  100.0% (200/200)
  Computation errors:    0

>>> ALL IMAGES BIT-EXACT MATCH — 硬件计算完全正确 <<<
>>> 真实 MNIST 手写数字识别成功！ <<<
